In [6]:
import pandas as pd

# =========================
# 1. Carregar dados
# =========================
vendas = pd.read_csv('../data/raw/vendas_2023_2024.csv')
custos = pd.read_csv('../data/processed/custos_importacao_normalizado.csv')

# =========================
# 2. Conversões
# =========================
vendas['sale_date'] = pd.to_datetime(
    vendas['sale_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

custos['start_date'] = pd.to_datetime(
    custos['start_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

# =========================
# 🔍 VALIDAÇÃO + FAIL FAST
# =========================

def validate_dates(df, column_name, dataset_name):
    invalid = df[column_name].isnull().sum()
    print(f"Datas inválidas em {dataset_name}: {invalid}")
    
    if invalid > 0:
        raise ValueError(f"Existem {invalid} datas inválidas em {dataset_name}.")

validate_dates(vendas, 'sale_date', 'vendas')
validate_dates(custos, 'start_date', 'custos')

# =========================
# Padronização de colunas
# =========================
vendas = vendas.rename(columns={'id_product': 'product_id'})

# print(vendas.columns)
# print(custos.columns)

# =========================
# Validação
# =========================
assert 'product_id' in vendas.columns, "Erro: vendas sem product_id"
assert 'product_id' in custos.columns, "Erro: custos sem product_id"

# =========================
# 3. Ordenação correta
# =========================
vendas = vendas.sort_values('sale_date')
custos = custos.sort_values('start_date')

# print(vendas[['product_id', 'sale_date']].head())
# print(custos[['product_id', 'start_date']].head())

# =========================
# 4. Merge temporal
# =========================
df = pd.merge_asof(
    vendas,
    custos,
    left_on='sale_date',
    right_on='start_date',
    by='product_id',
    direction='backward'
)

# =========================
# 5. Câmbio
# =========================
USD_TO_BRL = 5.0  # você pode justificar no relatório

df['cost_brl_unit'] = df['usd_price'] * USD_TO_BRL

# =========================
# 6. Cálculo correto de custo total
# =========================
df['cost_total'] = df['cost_brl_unit'] * df['qtd']

# =========================
# 7. Receita
# =========================
df['revenue'] = df['total']

# =========================
# 8. Lucro / Prejuízo
# =========================
df['profit'] = df['revenue'] - df['cost_total']

# =========================
# 9. Flag de prejuízo
# =========================
df['is_loss'] = df['profit'] < 0

# =========================
# 10. Agrupamento por produto
# =========================
prejuizo_produtos = (
    df.groupby('product_id')
    .agg(
        total_prejuizo=('profit', lambda x: x[x < 0].sum()),
        total_lucro=('profit', lambda x: x[x > 0].sum()),
        qtd_vendas=('product_id', 'count')
    )
    .sort_values(by='total_prejuizo')
)

# =========================
# 11. Resultado
# =========================
top_prejuizo = prejuizo_produtos.head(10)

print(top_prejuizo)

Datas inválidas em vendas: 0
Datas inválidas em custos: 0
            total_prejuizo  total_lucro  qtd_vendas
product_id                                         
72            -36177362.15         0.00          78
83            -16370851.50         0.00          44
74             -3890182.05    189945.20          85
71             -3840789.55   1923345.50          78
55             -3796577.75         0.00          74
91             -2265216.90    330206.10          63
78             -1794983.55    306325.80          61
60             -1709918.15   1801916.95          63
96             -1511014.90    240685.80          51
29             -1458259.15         0.00          70


In [13]:
# =========================
# Receita total
# =========================
receita_produtos = (
    df.groupby('product_id')['revenue']
    .sum()
    .rename('receita_total')
)

prejuizo_produtos = (
    prejuizo_produtos.drop(columns=['receita_total'], errors='ignore')
    .join(receita_produtos)
)

# =========================
# Percentual de perda
# =========================
prejuizo_produtos['percentual_perda'] = (
    prejuizo_produtos['total_prejuizo'] /
    prejuizo_produtos['receita_total']
)

# =========================
# Ordenação
# =========================
percentual = prejuizo_produtos.sort_values(by='percentual_perda')

# =========================
# FORMATAÇÃO
# =========================
def formatar_brl(valor):
    return f"R$ {valor:,.2f}"

def formatar_percentual(valor):
    return f"{valor * 100:.2f}%"

# Aplicar formatação
percentual_formatado = percentual.copy()

percentual_formatado['total_prejuizo'] = percentual_formatado['total_prejuizo'].apply(formatar_brl)
percentual_formatado['total_lucro'] = percentual_formatado['total_lucro'].apply(formatar_brl)
percentual_formatado['receita_total'] = percentual_formatado['receita_total'].apply(formatar_brl)
percentual_formatado['percentual_perda'] = percentual_formatado['percentual_perda'].apply(formatar_percentual)

percentual_formatado.head(5)

,total_prejuizo,total_lucro,qtd_vendas,percentual_perda,receita_total
product_id,,,,,
72,"R$ -36,177,362.15",R$ 0.00,78,-57.37%,"R$ 63,057,815.65"
83,"R$ -16,370,851.50",R$ 0.00,44,-36.89%,"R$ 44,377,440.00"
109,"R$ -88,826.10",R$ 622.80,67,-27.05%,"R$ 328,320.15"
102,"R$ -80,475.20",R$ 0.00,54,-24.00%,"R$ 335,379.20"
136,"R$ -231,383.80","R$ 2,305.70",69,-22.04%,"R$ 1,049,801.00"


In [9]:
top = percentual.iloc[0]

print(top)

total_prejuizo     -413534.350000
total_lucro         413840.350000
qtd_vendas              79.000000
percentual_perda     -1351.419444
receita_total          306.000000
Name: 18, dtype: float64
